# NLP Lab02:  Byte Pair Encoding

### BPE

## Setup

Only the standard library is needed for the three algorithms. `matplotlib` is used for the analysis plots, and `tokenizers` / `pythainlp` are optional extras for Sections 16–17.

In [1]:
# %pip install matplotlib

In [2]:
import re
import json
import math
import random
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Iterable, Optional

import matplotlib.pyplot as plt


try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("[note] `tokenizers` not installed -> Section 17 will be skipped.")
    print("       Install with:  pip install tokenizers")

print("Setup complete.")

Setup complete.


We build BPE first and in full detail. Parts II and III then reuse most of this machinery, so every function here is worth understanding properly.

In [3]:
train_text = "low low low low low lower lower newest newest newest widest widest"
train_vocab = set(train_text.split())

print("Word-level vocabulary learned from the training data:")
print(" ", sorted(train_vocab), "\n")

print("Now tokenize unseen text:")
for w in ["low", "lowest", "newer", "wider", "slowest"]:
    status = "ok" if w in train_vocab else "<UNK>   <-- information destroyed"
    print("  %-10s -> %s" % (w, status))

print("\nEvery <UNK> above is a word the model can never tell apart from any other")
print("unknown word -- even though 'lowest' obviously shares 'low' with a word we")
print("DID see five times.")

Word-level vocabulary learned from the training data:
  ['low', 'lower', 'newest', 'widest'] 

Now tokenize unseen text:
  low        -> ok
  lowest     -> <UNK>   <-- information destroyed
  newer      -> <UNK>   <-- information destroyed
  wider      -> <UNK>   <-- information destroyed
  slowest    -> <UNK>   <-- information destroyed

Every <UNK> above is a word the model can never tell apart from any other
unknown word -- even though 'lowest' obviously shares 'low' with a word we
DID see five times.


## A worked mini-example (do this by hand first)

Corpus, word -> frequency: `low` x5, `lower` x2, `newest` x6, `widest` x3.
We append `</w>` to mark the end of a word, so that `est` at the end of a word is a different symbol from `est` inside one.

| Step | Most frequent pair | Count | New symbol |
|---|---|---|---|
| start | `l o w </w>` · `l o w e r </w>` · `n e w e s t </w>` · `w i d e s t </w>` | | |
| 1 | `e` + `s` | 6 + 3 = 9 | `es` |
| 2 | `es` + `t` | 9 | `est` |
| 3 | `est` + `</w>` | 9 | `est</w>` |
| 4 | `l` + `o` | 5 + 2 = 7 | `lo` |
| 5 | `lo` + `w` | 7 | `low` |

Look at step 3: `est</w>` becomes a single token. BPE has discovered the English superlative suffix without being told that suffixes exist. Now let's make the computer do it.

In [4]:
END = "</w>"   # end-of-word marker

def pretokenize(text: str) -> List[str]:
    '''
    Split raw text into word-like chunks BEFORE any BPE happens.

    BPE is never applied across word boundaries, so this function decides what
    a 'word' is. Here: lowercase, then keep runs of letters, runs of digits,
    and single punctuation marks.
    '''
    return re.findall(r"[a-z]+|[0-9]+|[^\sa-z0-9]", text.lower())


def word_to_symbols(word: str) -> Tuple[str, ...]:
    '''Represent a word as its characters plus the end-of-word marker.'''
    return tuple(word) + (END,)


def build_corpus(text: str) -> Counter:
    '''Corpus = frequency table over words, each word held as a symbol tuple.'''
    return Counter(word_to_symbols(w) for w in pretokenize(text))

MINI_TEXT = "low " * 5 + "lower " * 2 + "newest " * 6 + "widest " * 3

print("MINI_TEXT: ", MINI_TEXT)
mini_corpus = build_corpus(MINI_TEXT)
for symbols, freq in mini_corpus.items():
    print("  %-30s x %d" % (" ".join(symbols), freq))

MINI_TEXT:  low low low low low lower lower newest newest newest newest newest newest widest widest widest 
  l o w </w>                     x 5
  l o w e r </w>                 x 2
  n e w e s t </w>               x 6
  w i d e s t </w>               x 3


## Step 1 — count adjacent pairs

In [5]:
def get_pair_freqs(corpus: Counter) -> Dict[Tuple[str, str], int]:
    '''Count every adjacent symbol pair in the corpus, weighted by word frequency.'''
    pair_freqs = defaultdict(int)
    for symbols, freq in corpus.items():
        # print("symbols, freq", symbols, freq)
        for i in range(len(symbols) - 1):
            pair_freqs[(symbols[i], symbols[i + 1])] += freq
        #     print(pair_freqs)
        # print("------------")
    return dict(pair_freqs)


pair_freqs = get_pair_freqs(mini_corpus)

print("Top pairs in the mini-corpus:")
for pair, freq in sorted(pair_freqs.items(), key=lambda kv: -kv[1])[:6]:
    print("  %-20s -> %d" % (str(pair), freq))

print("\nMost frequent pair:", max(pair_freqs, key=pair_freqs.get))

Top pairs in the mini-corpus:
  ('e', 's')           -> 9
  ('s', 't')           -> 9
  ('t', '</w>')        -> 9
  ('w', 'e')           -> 8
  ('l', 'o')           -> 7
  ('o', 'w')           -> 7

Most frequent pair: ('e', 's')


## Step 2 — apply one merge

Merging the pair `(a, b)` means: scan every word left to right and replace each occurrence of `a` immediately followed by `b` with the single symbol `a+b`.

Two details that bite people:

* **Scan left to right, non-overlapping.** In `a a a`, merging `(a, a)` gives `aa a` — not `aa aa`.
* **Accumulate with `+=`, do not assign.** Two different words can collapse to the same symbol tuple after a merge, and overwriting the entry would silently lose a frequency count.

In [6]:
def merge_symbols(symbols: Tuple[str, ...], pair: Tuple[str, str]) -> Tuple[str, ...]:
    '''Replace every non-overlapping occurrence of `pair` in one word, left to right.'''
    a, b = pair
    merged = a + b
    out: List[str] = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            out.append(merged)
            i += 2          # skip both symbols we just consumed
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)


def merge_pair(corpus: Counter, pair: Tuple[str, str]) -> Counter:
    '''Apply one merge rule to the whole corpus, returning a NEW corpus.'''
    new_corpus = Counter()
    for symbols, freq in corpus.items():
        new_corpus[merge_symbols(symbols, pair)] += freq   # += , not =
    return new_corpus


# --- watch the first three merges happen ---
corpus_demo = mini_corpus.copy()
for step in range(1, 4):
    freqs = get_pair_freqs(corpus_demo)
    # highest count wins; ties go to the alphabetically smaller pair (see Section 6)
    best = min(freqs, key=lambda p: (-freqs[p], p))
    print("merge %d: %r + %r -> %r   (count=%d)"
          % (step, best[0], best[1], "".join(best), freqs[best]))
    corpus_demo = merge_pair(corpus_demo, best)
    for symbols, freq in corpus_demo.items():
        print("      %-28s x %d" % (" ".join(symbols), freq))
    print()

merge 1: 'e' + 's' -> 'es'   (count=9)
      l o w </w>                   x 5
      l o w e r </w>               x 2
      n e w es t </w>              x 6
      w i d es t </w>              x 3

merge 2: 'es' + 't' -> 'est'   (count=9)
      l o w </w>                   x 5
      l o w e r </w>               x 2
      n e w est </w>               x 6
      w i d est </w>               x 3

merge 3: 'est' + '</w>' -> 'est</w>'   (count=9)
      l o w </w>                   x 5
      l o w e r </w>               x 2
      n e w est</w>                x 6
      w i d est</w>                x 3



## Step 3 — the training loop

Now just repeat. Three design decisions worth saying out loud:

1. **Tie-breaking must be deterministic.** Ties are common — in the mini-corpus, three different pairs all occur 9 times. Plain `max()` returns whichever came first in iteration order, which can vary between runs and across Python versions. We sort by `(-count, pair)`: highest count first, ties going to the alphabetically smaller pair. Training twice then gives the same tokenizer twice.
2. **Stopping early.** If the best remaining pair occurs once, merging it only memorises one word. `min_freq` lets us stop.
3. **The vocabulary grows by exactly one token per merge.** So `final vocab size = number of distinct characters + num_merges` — that is how you hit a target vocabulary size (Exercise 2).

In [7]:
def train_bpe(corpus: Counter,
              num_merges: int,
              min_freq: int = 1,
              verbose: bool = True) -> Tuple[List[Tuple[str, str]], List[str], List[dict]]:
    '''
    Learn BPE merge rules from a corpus.

    Args:
        corpus:     Counter mapping symbol-tuples -> word frequency
        num_merges: how many merge operations to learn
        min_freq:   stop early if the best pair is rarer than this
        verbose:    print every merge

    Returns:
        merges:  ordered list of (a, b) merge rules  <- the trained model
        vocab:   list of every token string, base characters first
        history: per-step statistics, used for the analysis in Section 9
    '''
    corpus = Counter(corpus)                 # work on a copy

    # base vocabulary = every symbol that appears anywhere (characters + </w>)
    base_symbols = sorted({s for symbols in corpus for s in symbols})
    vocab: List[str] = list(base_symbols)

    merges: List[Tuple[str, str]] = []
    history: List[dict] = []

    for step in range(1, num_merges + 1):
        pair_freqs = get_pair_freqs(corpus)
        if not pair_freqs:
            print("[stop] nothing left to merge after %d merges." % (step - 1))
            break

        # deterministic tie-break: highest count first, then smallest pair
        best = min(pair_freqs, key=lambda p: (-pair_freqs[p], p))
        best_freq = pair_freqs[best]

        if best_freq < min_freq:
            print("[stop] best pair occurs only %dx (< min_freq=%d)." % (best_freq, min_freq))
            break

        corpus = merge_pair(corpus, best)
        merges.append(best)
        vocab.append("".join(best))

        history.append({
            "step": step,
            "pair": best,
            "freq": best_freq,
            "new_token": "".join(best),
            "vocab_size": len(vocab),
            "corpus_tokens": sum(len(s) * f for s, f in corpus.items()),
        })

        if verbose:
            print("  merge %3d: %10r + %-10r -> %-14r (count=%d)"
                  % (step, best[0], best[1], "".join(best), best_freq))

    if verbose:
        print("\nDone: %d merges, vocabulary size = %d (%d base symbols + %d merged tokens)"
              % (len(merges), len(vocab), len(base_symbols), len(merges)))
    return merges, vocab, history


mini_merges, mini_vocab, _ = train_bpe(mini_corpus, num_merges=10)
print("\nMerge rules learned:", mini_merges)

  merge   1:        'e' + 's'        -> 'es'           (count=9)
  merge   2:       'es' + 't'        -> 'est'          (count=9)
  merge   3:      'est' + '</w>'     -> 'est</w>'      (count=9)
  merge   4:        'l' + 'o'        -> 'lo'           (count=7)
  merge   5:       'lo' + 'w'        -> 'low'          (count=7)
  merge   6:        'e' + 'w'        -> 'ew'           (count=6)
  merge   7:       'ew' + 'est</w>'  -> 'ewest</w>'    (count=6)
  merge   8:        'n' + 'ewest</w>' -> 'newest</w>'   (count=6)
  merge   9:      'low' + '</w>'     -> 'low</w>'      (count=5)
  merge  10:        'd' + 'est</w>'  -> 'dest</w>'     (count=3)

Done: 10 merges, vocabulary size = 21 (11 base symbols + 10 merged tokens)

Merge rules learned: [('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o'), ('lo', 'w'), ('e', 'w'), ('ew', 'est</w>'), ('n', 'ewest</w>'), ('low', '</w>'), ('d', 'est</w>')]


Compare the first five printed merges with the hand-worked table in Section 2: `e`+`s`, `es`+`t`, `est`+`</w>`, `l`+`o`, `lo`+`w`. The implementation reproduces the trace exactly — including the three-way tie at count 9, which the tie-break rule resolves in favour of `('e', 's')`.

Merges 6 onward go past the table and show what happens when a corpus is tiny: by merge 8 the rules have glued an entire word together into the single token `newest</w>`. With only four word types there is nothing left to generalise over — which is the whole argument for training a tokenizer on a large corpus.

Now train on something larger, so the vocabulary gets interesting.

In [8]:
CORPUS_TEXT = '''
Byte pair encoding is a simple form of data compression in which the most common
pair of consecutive bytes of data is replaced with a byte that does not occur
within that data. Tokenization is the process of splitting text into smaller
units called tokens. Subword tokenization helps handle out of vocabulary words
by breaking rare words into known subword units. Natural language processing
requires an efficient text representation. Deep learning models benefit from
subword tokenization schemes because the embedding matrix stays small.
Neural machine translation systems use byte pair encoding to handle rare words,
names, and morphologically rich languages. A tokenizer is trained once on a
large corpus and then reused unchanged by every model that shares it.

The lowest layer of a language model is an embedding lookup, so the tokenizer
decides what the model is even able to see. Tokenizers that split words into
smaller pieces make sequences longer but vocabularies smaller. Longer sequences
cost more attention computation; larger vocabularies cost more embedding
parameters. Choosing a vocabulary size is therefore a trade off between these
two costs. Frequent words such as the, of, and, to, is, in, that, it, for, was,
end up as single tokens after training, while rare words such as tokenizer,
morphologically, and hyperparameter are split into several pieces.

Character level models never produce unknown tokens, but their sequences are
far too long. Word level models produce short sequences, but they cannot read
any word they did not see during training. Subword models sit between these
two extremes, which is why nearly every modern language model uses one.

A tokenizer is trained by counting. The trainer reads the training corpus,
counts how often each pattern occurs, and keeps the patterns that occur often
enough to be worth a vocabulary entry. Counting is cheap, and the counting is
done over word types rather than word tokens, so training a tokenizer on a
very large corpus is much faster than training the model that will use it.

Tokenizing text is not the same as understanding text. A tokenizer knows
nothing about meaning, grammar, or context. It knows only which sequences of
characters occurred together often in the training corpus. That is enough to
produce useful units, because frequent character sequences in a language tend
to be morphemes, common words, and common word endings.

Consider the words tokenize, tokenizer, tokenizing, tokenized, and tokenization.
A word level tokenizer stores five unrelated entries, and the model must learn
five unrelated embeddings. A subword tokenizer stores the shared stem once and
adds short endings, so the model sees immediately that the five words are
related. This sharing is most valuable in languages with rich morphology, where
a single stem can appear in dozens of inflected forms.

Normalization happens before tokenization. A normalizer lowercases text,
strips accents, and replaces unusual characters with standard ones. Different
models normalize differently, and two tokenizers trained on the same corpus
with different normalizers will produce different vocabularies. Normalization
is destructive: an uppercase letter that has been lowercased cannot be
recovered, so a model trained on normalized text cannot reproduce case.

Special tokens are added to the vocabulary by hand rather than learned from
the corpus. A padding token fills short sequences so that a batch is
rectangular. An unknown token stands in for text the tokenizer cannot
represent. A classification token collects a representation of a whole
sequence. A separator token marks the boundary between two segments. These
tokens are never produced by the training algorithm; they are reserved before
training begins.

Tokenizer choices have consequences that surface much later. A vocabulary that
handles English well may split another language into single characters, making
sequences in that language several times longer and effectively shrinking the
context window for its speakers. Numbers are another common failure: a
tokenizer that splits numbers inconsistently makes arithmetic harder for the
model to learn, so many recent models tokenize every digit separately on
purpose.

Evaluating a tokenizer is mostly a matter of measuring compression on text
that was not used for training. Fewer tokens per word means more text fits in
the context window and fewer embedding lookups are needed per sentence. But
compression is not the only goal: a tokenizer that produces pieces aligned
with morphemes gives the model a more useful signal than one that produces
arbitrary frequent fragments of equal length.

Training a tokenizer is fast, but changing one later is expensive. Every
embedding in the model is tied to a vocabulary entry, so replacing the
tokenizer means retraining the model. For this reason the vocabulary is
usually chosen early, reused across an entire family of models, and shared
between the pretraining corpus and every downstream application.
'''

corpus = build_corpus(CORPUS_TEXT)
print("%d word tokens, %d word types, %d distinct characters\n"
      % (sum(corpus.values()), len(corpus), len({s for w in corpus for s in w})))

merges, vocab, history = train_bpe(corpus, num_merges=200, verbose=False)
print("Learned %d merges -> vocabulary of %d tokens\n" % (len(merges), len(vocab)))

print("First 15 merges (the most frequent patterns in this corpus):")
for h in history[:15]:
    print("  %3d. %r + %r -> %r (count=%d)"
          % (h["step"], h["pair"][0], h["pair"][1], h["new_token"], h["freq"]))

print("\nLongest tokens learned:")
for tok_str in sorted(vocab, key=len, reverse=True)[:12]:
    print("  ", repr(tok_str))

875 word tokens, 342 word types, 30 distinct characters

Learned 200 merges -> vocabulary of 230 tokens

First 15 merges (the most frequent patterns in this corpus):
    1. 's' + '</w>' -> 's</w>' (count=154)
    2. 'e' + '</w>' -> 'e</w>' (count=122)
    3. 'e' + 'n' -> 'en' (count=97)
    4. 'i' + 'n' -> 'in' (count=87)
    5. 't' + 'h' -> 'th' (count=84)
    6. 'e' + 'r' -> 'er' (count=83)
    7. 't' + '</w>' -> 't</w>' (count=76)
    8. 'o' + 'r' -> 'or' (count=63)
    9. 'd' + '</w>' -> 'd</w>' (count=62)
   10. 'a' + 'n' -> 'an' (count=56)
   11. 't' + 'o' -> 'to' (count=55)
   12. 'a' + 'r' -> 'ar' (count=54)
   13. 'er' + '</w>' -> 'er</w>' (count=49)
   14. 'o' + 'n' -> 'on' (count=48)
   15. 'in' + 'g' -> 'ing' (count=47)

Longest tokens learned:
   'tokenization</w>'
   'vocabulary</w>'
   'tokenizer</w>'
   'sequences</w>'
   'embedding</w>'
   'training</w>'
   'language</w>'
   'produce</w>'
   'subword</w>'
   'between</w>'
   'corpus</w>'
   'models</w>'


## Step 4 — encoding new text

Training is only half a tokenizer. To encode **unseen** text we:

1. pre-tokenize it exactly the same way as during training,
2. split each word into characters + `</w>`,
3. replay the merge rules **in the order they were learned**.

In [9]:
def encode_word_naive(word: str, merges: List[Tuple[str, str]]) -> List[str]:
    '''Reference encoder: replay every merge rule, in learned order.'''
    symbols = word_to_symbols(word)
    for pair in merges:
        if len(symbols) == 1:
            break
        symbols = merge_symbols(symbols, pair)
    return list(symbols)


def build_ranks(merges: List[Tuple[str, str]]) -> Dict[Tuple[str, str], int]:
    '''rank[(a, b)] = position of the rule in training order (lower = learned earlier).'''
    return {pair: i for i, pair in enumerate(merges)}


def encode_word(word: str,
                ranks: Dict[Tuple[str, str], int],
                cache: Dict[str, List[str]] = None) -> List[str]:
    '''Fast encoder: repeatedly apply the highest-priority applicable merge.'''
    if cache is not None and word in cache:
        return cache[word]

    symbols = word_to_symbols(word)
    while len(symbols) > 1:
        # among the pairs present in this word, take the one learned earliest
        candidates = [(ranks[p], p) for p in zip(symbols, symbols[1:]) if p in ranks]
        if not candidates:
            break
        symbols = merge_symbols(symbols, min(candidates)[1])

    tokens = list(symbols)
    if cache is not None:
        cache[word] = tokens
    return tokens


def encode(text: str, ranks: Dict[Tuple[str, str], int], cache=None) -> List[str]:
    '''Tokenize a whole string into subword tokens.'''
    return [t for w in pretokenize(text) for t in encode_word(w, ranks, cache)]


ranks = build_ranks(merges)
cache: Dict[str, List[str]] = {}

# the two encoders must agree on every word
for w in pretokenize(CORPUS_TEXT) + ["hyperparameter", "untokenizable", "zzz"]:
    assert encode_word_naive(w, merges) == encode_word(w, ranks), w
print("OK: the naive encoder and the rank-based encoder agree on every word.\n")

for sentence in ["Tokenization helps neural networks.",
                 "BPE handles unknownword gracefully.",
                 "The lowest layer is an embedding lookup."]:
    toks = encode(sentence, ranks, cache)
    print("text   :", sentence)
    print("tokens :", toks)
    print("count  : %d tokens\n" % len(toks))

OK: the naive encoder and the rank-based encoder agree on every word.

text   : Tokenization helps neural networks.
tokens : ['tokenization</w>', 'h', 'e', 'l', 'p', 's</w>', 'n', 'e', 'ur', 'al</w>', 'n', 'e', 't', 'wor', 'k', 's</w>', '.</w>']
count  : 17 tokens

text   : BPE handles unknownword gracefully.
tokens : ['b', 'p', 'e</w>', 'h', 'and', 'l', 'es</w>', 'un', 'know', 'n', 'word</w>', 'g', 'r', 'ac', 'e', 'f', 'u', 'l', 'ly</w>', '.</w>']
count  : 20 tokens

text   : The lowest layer is an embedding lookup.
tokens : ['the</w>', 'lo', 'w', 'e', 'st</w>', 'l', 'a', 'y', 'er</w>', 'is</w>', 'an</w>', 'embedding</w>', 'lo', 'o', 'k', 'u', 'p', '</w>', '.</w>']
count  : 19 tokens



In [10]:
print("%18s | %18s | tokens" % ("word", "seen in training?"))
print("-" * 78)
seen_words = set(pretokenize(CORPUS_TEXT))
for w in ["tokenization", "tokenizer", "detokenizing", "supercalifragilistic", "xylophone"]:
    mark = "yes" if w in seen_words else "NO (unseen)"
    print("%18s | %18s | %s" % (w, mark, encode_word(w, ranks)))

print("\nFewer tokens = the tokenizer recognised the word.")
print("More tokens  = it had to spell the word out from smaller pieces.")

              word |  seen in training? | tokens
------------------------------------------------------------------------------
      tokenization |                yes | ['tokenization</w>']
         tokenizer |                yes | ['tokenizer</w>']
      detokenizing |        NO (unseen) | ['d', 'e', 'tokeniz', 'ing</w>']
supercalifragilistic |        NO (unseen) | ['su', 'p', 'er', 'ca', 'l', 'i', 'f', 'r', 'ag', 'il', 'i', 'st', 'ic', '</w>']
         xylophone |        NO (unseen) | ['x', 'y', 'lo', 'p', 'h', 'on', 'e</w>']

Fewer tokens = the tokenizer recognised the word.
More tokens  = it had to spell the word out from smaller pieces.


## Step 5 — decoding, and the token-id vocabulary

In [11]:
def decode(tokens: Iterable[str]) -> str:
    '''Invert `encode`: join the tokens and restore word boundaries.'''
    return "".join(tokens).replace(END, " ").strip()


class BPETokenizer:
    '''A minimal but complete tokenizer: train, encode, decode, save, load.'''

    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, merges: List[Tuple[str, str]], vocab: List[str]):
        self.merges = [tuple(m) for m in merges]
        self.ranks = build_ranks(self.merges)
        self.itos = [self.PAD, self.UNK] + list(vocab)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._cache: Dict[str, List[str]] = {}

    # ---- construction --------------------------------------------------
    @classmethod
    def train(cls, text: str, num_merges: int, min_freq: int = 1, verbose: bool = False):
        m, v, _ = train_bpe(build_corpus(text), num_merges, min_freq, verbose)
        return cls(m, v)

    # ---- the tokenizer API ---------------------------------------------
    def tokenize(self, text: str) -> List[str]:
        return encode(text, self.ranks, self._cache)

    def encode(self, text: str) -> List[int]:
        unk = self.stoi[self.UNK]
        return [self.stoi.get(t, unk) for t in self.tokenize(text)]

    def decode(self, ids: Iterable[int]) -> str:
        pad = self.stoi[self.PAD]
        return decode(self.itos[i] for i in ids if i != pad)

    # ---- persistence ----------------------------------------------------
    def save(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "vocab": self.itos[2:]}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path: str):
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        return cls(obj["merges"], obj["vocab"])

    def __len__(self):
        return len(self.itos)


tok = BPETokenizer(merges, vocab)
sample = "Subword tokenization helps rare words."
ids = tok.encode(sample)

print("vocabulary size:", len(tok))
print("\ntext    :", sample)
print("tokens  :", tok.tokenize(sample))
print("ids     :", ids)
print("decoded :", repr(tok.decode(ids)))
print("\nDecoding recovers the pre-tokenized text: lowercased, with punctuation")
print("separated -- exactly the text BPE was given, not the original string.")

vocabulary size: 232

text    : Subword tokenization helps rare words.
tokens  : ['subword</w>', 'tokenization</w>', 'h', 'e', 'l', 'p', 's</w>', 'r', 'are</w>', 'words</w>', '.</w>']
ids     : [206, 208, 14, 11, 17, 21, 32, 23, 106, 136, 48]
decoded : 'subword tokenization helps rare words .'

Decoding recovers the pre-tokenized text: lowercased, with punctuation
separated -- exactly the text BPE was given, not the original string.


In [12]:
# A trained tokenizer is just a list of rules plus a vocabulary -- save and reload it.
tok.save("bpe_tokenizer.json")
reloaded = BPETokenizer.load("bpe_tokenizer.json")

assert reloaded.encode(sample) == ids
print("Saved to bpe_tokenizer.json and reloaded -> identical output.\n")
print("File contents (first 200 characters):")
print(open("bpe_tokenizer.json", encoding="utf-8").read()[:200], "...")

Saved to bpe_tokenizer.json and reloaded -> identical output.

File contents (first 200 characters):
{"merges": [["s", "</w>"], ["e", "</w>"], ["e", "n"], ["i", "n"], ["t", "h"], ["e", "r"], ["t", "</w>"], ["o", "r"], ["d", "</w>"], ["a", "n"], ["t", "o"], ["a", "r"], ["er", "</w>"], ["o", "n"], ["in ...


## Thai Tokenization

In [13]:
THAI_TEXT = '''
การประมวลผลภาษาธรรมชาติเป็นสาขาหนึ่งของปัญญาประดิษฐ์
ภาษาไทยเขียนติดกันโดยไม่มีช่องว่างระหว่างคำ
การตัดคำภาษาไทยจึงเป็นปัญหาที่ยากกว่าภาษาอังกฤษ
การประมวลผลภาษาไทยต้องอาศัยการตัดคำที่ถูกต้อง
แบบจำลองภาษาขนาดใหญ่ใช้การตัดคำย่อยเพื่อจัดการคำที่พบไม่บ่อย
การตัดคำย่อยช่วยลดปัญหาคำที่ไม่เคยพบในชุดข้อมูลฝึกสอน
ปัญญาประดิษฐ์และการประมวลผลภาษาธรรมชาติเติบโตอย่างรวดเร็ว
การประมวลผลภาษาธรรมชาติใช้แบบจำลองภาษาขนาดใหญ่
'''


def pretokenize_thai(text: str) -> List[str]:
    '''Split on whitespace only -- each chunk is a phrase, not a word.'''
    return text.split()

# same machinery as before; only the pre-tokenizer changed
thai_corpus = Counter(word_to_symbols(w) for w in pretokenize_thai(THAI_TEXT))
thai_merges, thai_vocab, _ = train_bpe(thai_corpus, num_merges=80, verbose=False)
thai_ranks = build_ranks(thai_merges)

print("Subwords that BPE discovered in Thai (longest first):")
for t in sorted((t for t in thai_vocab if len(t) > 2 and END not in t), key=len, reverse=True)[:12]:
    print("  ", t)

test_th = "การประมวลผลภาษาไทยด้วยปัญญาประดิษฐ์"
print("\ntext   :", test_th)
print("BPE    :", encode_word(test_th, thai_ranks))

Subwords that BPE discovered in Thai (longest first):
   การประมวลผลภาษาธรรมชาติเ
   การประมวลผลภาษาธรรมชาติ
   การประมวลผลภาษาธรรมชา
   การประมวลผลภาษาธรรมช
   แบบจำลองภาษาขนาดใหญ่
   การประมวลผลภาษาธรรม
   บบจำลองภาษาขนาดใหญ่
   การประมวลผลภาษาธรร
   บจำลองภาษาขนาดใหญ่
   การประมวลผลภาษาธร
   จำลองภาษาขนาดใหญ่
   การประมวลผลภาษาธ

text   : การประมวลผลภาษาไทยด้วยปัญญาประดิษฐ์
BPE    : ['การประมวลผลภาษา', 'ไทย', 'ด', '้', 'ว', 'ย', 'ปัญญาประดิษฐ์', '</w>']


In [14]:
# %pip install pythainlp

In [15]:
try:
    from pythainlp.tokenize import word_tokenize

    thai_words = [w for w in word_tokenize(THAI_TEXT, engine="newmm") if w.strip()]
    print("PyThaiNLP segmentation (first 20 words):", thai_words[:20], "\n")

    seg_corpus = Counter(word_to_symbols(w) for w in thai_words)
    seg_merges, seg_vocab, _ = train_bpe(seg_corpus, num_merges=80, verbose=False)
    seg_ranks = build_ranks(seg_merges)

    print("segmenter + BPE on the test phrase:")
    for w in word_tokenize(test_th, engine="newmm"):
        if w.strip():
            print("  %10s -> %s" % (w, encode_word(w, seg_ranks)))
except ImportError:
    print("[skipped] PyThaiNLP not installed.  pip install pythainlp")

PyThaiNLP segmentation (first 20 words): ['การประมวลผล', 'ภาษาธรรมชาติ', 'เป็น', 'สาขา', 'หนึ่ง', 'ของ', 'ปัญญาประดิษฐ์', 'ภาษาไทย', 'เขียน', 'ติดกัน', 'โดย', 'ไม่', 'มี', 'ช่องว่าง', 'ระหว่าง', 'คำ', 'การ', 'ตัด', 'คำ', 'ภาษาไทย'] 

segmenter + BPE on the test phrase:
  การประมวลผล -> ['การประมวลผล</w>']
     ภาษาไทย -> ['ภาษาไทย</w>']
        ด้วย -> ['ด', '้', 'ว', 'ย</w>']
  ปัญญาประดิษฐ์ -> ['ปัญญาประดิษฐ์</w>']


# NLP Lab 02: Word Segmentation, POS Tagging, and Sequence Labeling
**Course:** Natural Language Processing

---
# Part 1 — Word Segmentation

## 1.1 Why segmentation is a *task*

Two families of solutions:

| Approach | Idea | Needs |
|---|---|---|
| **Dictionary / rule-based** (§1.2) | Match the longest string that is in a word list | A dictionary |
| **Statistical / learned** (§1.6) | Choose the segmentation with the highest probability | A corpus (counts), or a trained model |

## 1.2 Maximum Matching (MaxMatch)

**Maximum Matching** is a greedy algorithm: repeatedly bite off the *longest* string that appears in the dictionary. It comes in two directions.

**Forward Maximum Matching (FMM)** — scan left to right:

**Backward Maximum Matching (BMM)** — the same thing from the right end:

In [16]:
# ---- A tiny Thai dictionary (word list) -------------------------------------
# In practice you'd load ~60k words (e.g. PyThaiNLP's 'newmm' dictionary).
# We keep it small so every matching step is easy to follow by hand.
THAI_DICT = {
    "ตา",        # eye
    "ตาก",       # to air out / dry in the sun
    "กลม",       # round
    "ลม",        # wind
    "นอน",       # to lie down / sleep
    "หลับ",      # asleep
    "นอนหลับ",   # to sleep (compound)
    "อากาศ",     # weather / air
    "นั่ง",       # to sit
    "ริม",        # edge / bank
    "ทะเล",      # sea
}


def forward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Forward Maximum Matching (FMM): greedy longest match, left to right."""
    tokens, steps = [], []
    i = 0
    while i < len(text):
        matched = None
        for L in range(min(max_word_len, len(text) - i), 0, -1):
            candidate = text[i:i + L]
            if candidate in dictionary:
                matched = candidate
                steps.append((i, candidate, True))
                break
            steps.append((i, candidate, False))
        if matched is None:                 # unknown character: emit it alone
            matched = text[i]
            steps.append((i, matched, None))
        tokens.append(matched)
        i += len(matched)

    if verbose:
        print(f"FMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   i={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens


def backward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Backward Maximum Matching (BMM): greedy longest match, right to left."""
    tokens, steps = [], []
    j = len(text)
    while j > 0:
        matched = None
        for L in range(min(max_word_len, j), 0, -1):
            candidate = text[j - L:j]
            if candidate in dictionary:
                matched = candidate
                steps.append((j, candidate, True))
                break
            steps.append((j, candidate, False))
        if matched is None:
            matched = text[j - 1]
            steps.append((j, matched, None))
        tokens.insert(0, matched)
        j -= len(matched)

    if verbose:
        print(f"BMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   j={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens


print("Dictionary size:", len(THAI_DICT), "words")

Dictionary size: 11 words


## 1.3 MaxMatch on English 

In [17]:
ENG_DICT = {"the", "cat", "in", "hat", "table", "down", "there",
            "theta", "bled", "own", "a", "i"}

print("Easy case:")
forward_max_match("thecatinthehat", ENG_DICT, max_word_len=6, verbose=True)

Easy case:
FMM on 'thecatinthehat'
   i= 0  try 'thecat'     (len 6)  not in dict
   i= 0  try 'theca'      (len 5)  not in dict
   i= 0  try 'thec'       (len 4)  not in dict
   i= 0  try 'the'        (len 3)  MATCH -> emit
   i= 3  try 'catint'     (len 6)  not in dict
   i= 3  try 'catin'      (len 5)  not in dict
   i= 3  try 'cati'       (len 4)  not in dict
   i= 3  try 'cat'        (len 3)  MATCH -> emit
   i= 6  try 'intheh'     (len 6)  not in dict
   i= 6  try 'inthe'      (len 5)  not in dict
   i= 6  try 'inth'       (len 4)  not in dict
   i= 6  try 'int'        (len 3)  not in dict
   i= 6  try 'in'         (len 2)  MATCH -> emit
   i= 8  try 'thehat'     (len 6)  not in dict
   i= 8  try 'theha'      (len 5)  not in dict
   i= 8  try 'theh'       (len 4)  not in dict
   i= 8  try 'the'        (len 3)  MATCH -> emit
   i=11  try 'hat'        (len 3)  MATCH -> emit
   result: the|cat|in|the|hat



['the', 'cat', 'in', 'the', 'hat']

In [18]:
forward_max_match("thetabledownthere", ENG_DICT, max_word_len=6, verbose=True)

FMM on 'thetabledownthere'
   i= 0  try 'thetab'     (len 6)  not in dict
   i= 0  try 'theta'      (len 5)  MATCH -> emit
   i= 5  try 'bledow'     (len 6)  not in dict
   i= 5  try 'bledo'      (len 5)  not in dict
   i= 5  try 'bled'       (len 4)  MATCH -> emit
   i= 9  try 'ownthe'     (len 6)  not in dict
   i= 9  try 'ownth'      (len 5)  not in dict
   i= 9  try 'ownt'       (len 4)  not in dict
   i= 9  try 'own'        (len 3)  MATCH -> emit
   i=12  try 'there'      (len 5)  MATCH -> emit
   result: theta|bled|own|there



['theta', 'bled', 'own', 'there']

In [19]:
backward_max_match("thetabledownthere", ENG_DICT, max_word_len=6, verbose=True)

BMM on 'thetabledownthere'
   j=17  try 'nthere'     (len 6)  not in dict
   j=17  try 'there'      (len 5)  MATCH -> emit
   j=12  try 'ledown'     (len 6)  not in dict
   j=12  try 'edown'      (len 5)  not in dict
   j=12  try 'down'       (len 4)  MATCH -> emit
   j= 8  try 'etable'     (len 6)  not in dict
   j= 8  try 'table'      (len 5)  MATCH -> emit
   j= 3  try 'the'        (len 3)  MATCH -> emit
   result: the|table|down|there



['the', 'table', 'down', 'there']

### What just happened

| Input | FMM output | Correct? |
|---|---|---|
| `thecatinthehat` | `the | cat | in | the | hat` | ✅ |
| `thetabledownthere` | `theta | bled | own | there` | ❌ (should be *the table down there*) |
| `thetabledownthere` (BMM) | `the | table | down | there` | ✅ |


## 1.4 The core Thai ambiguity: **ตากลม**

The 5-character string **ตากลม** (`ต า ก ล ม`) has two completely valid readings, depending on where you put the single boundary:

| Segmentation | Words | Meaning |
|---|---|---|
| **ตา | กลม** | ตา (eye) + กลม (round) | "round eyes" / round-eyed |
| **ตาก | ลม** | ตาก (to air out) + ลม (wind) | "to air out in the wind" |

In [20]:
AMBIGUOUS = "ตากลม"

fmm_result = forward_max_match(AMBIGUOUS, THAI_DICT, verbose=True)
bmm_result = backward_max_match(AMBIGUOUS, THAI_DICT, verbose=True)


FMM on 'ตากลม'
   i= 0  try 'ตากลม'      (len 5)  not in dict
   i= 0  try 'ตากล'       (len 4)  not in dict
   i= 0  try 'ตาก'        (len 3)  MATCH -> emit
   i= 3  try 'ลม'         (len 2)  MATCH -> emit
   result: ตาก|ลม

BMM on 'ตากลม'
   j= 5  try 'ตากลม'      (len 5)  not in dict
   j= 5  try 'ากลม'       (len 4)  not in dict
   j= 5  try 'กลม'        (len 3)  MATCH -> emit
   j= 2  try 'ตา'         (len 2)  MATCH -> emit
   result: ตา|กลม



## 1.5 Step-by-step: exactly where they diverge

**FMM** (scanning left to right from `i=0`):

| Step | Position | Candidate tried | In dictionary? | Action |
|---|---|---|---|---|
| 1 | i=0 | ตากลม (5) | ✗ | shrink |
| 2 | i=0 | ตากล (4) | ✗ | shrink |
| 3 | i=0 | **ตาก** (3) | ✓ | **emit ตาก**, i → 3 |
| 4 | i=3 | **ลม** (2) | ✓ | **emit ลม**, i → 5 = end |

→ **ตาก | ลม**

**BMM** (scanning right to left from `j=5`):

| Step | Position | Candidate tried | In dictionary? | Action |
|---|---|---|---|---|
| 1 | j=5 | ตากลม (5) | ✗ | shrink |
| 2 | j=5 | ากลม (4) | ✗ | shrink |
| 3 | j=5 | **กลม** (3) | ✓ | **emit กลม**, j → 2 |
| 4 | j=2 | **ตา** (2) | ✓ | **emit ตา**, j → 0 = end |

→ **ตา | กลม**

## 1.6 Statistical segmentation

Reframe segmentation as **search over all possible segmentations**, scored by a language model.

In [21]:
import math

# ---- toy corpus counts (illustrative, N = 1,000,000 tokens) -----------------
N_TOKENS = 1000000 
UNIGRAM_COUNTS = {
    "ตา":     4200,   # eye        - common
    "กลม":    1800,   # round      - common
    "ตาก":     320,   # to air out - rarer
    "ลม":     3600,   # wind       - common
    "นั่ง":    2000,   # to sit
    "นอน":    5100,
    "หลับ":   1400,
    "นอนหลับ": 900,
    "อากาศ":  2900,
}

def unigram_prob(word):
    return UNIGRAM_COUNTS.get(word, 1) / N_TOKENS      # crude add-one-ish floor

def score_segmentation(tokens):
    """Return (probability, log10 probability) of a segmentation under the unigram LM."""
    p = 1.0
    for w in tokens:
        p *= unigram_prob(w)
    return p, math.log10(p)

# 2 candidates to find prob
cand_A = ["ตา", "กลม"]     # BMM's answer: 'round eyes'
cand_B = ["ตาก", "ลม"]     # FMM's answer: 'airing in the wind'

for name, cand in [("A (ตา|กลม)", cand_A), ("B (ตาก|ลม)", cand_B)]:
    parts = "  x  ".join(f"P({w})={unigram_prob(w):.5f}" for w in cand)
    p, logp = score_segmentation(cand)
    print(f"{name:<14} {parts}   =  {p:.4e}   (log10 = {logp:.4f})")

pA, _ = score_segmentation(cand_A)
pB, _ = score_segmentation(cand_B)
winner = cand_A if pA > pB else cand_B
print(f"\nWinner (unigram): {'|'.join(winner)}   by a factor of {max(pA, pB) / min(pA, pB):.4f}x")

A (ตา|กลม)     P(ตา)=0.00420  x  P(กลม)=0.00180   =  7.5600e-06   (log10 = -5.1215)
B (ตาก|ลม)     P(ตาก)=0.00032  x  P(ลม)=0.00360   =  1.1520e-06   (log10 = -5.9385)

Winner (unigram): ตา|กลม   by a factor of 6.5625x


### The arithmetic, written out

With `N = 1,000,000`:

- `P(ตา)  = 4200 / 1,000,000  = 0.00420`
- `P(กลม) = 1800 / 1,000,000  = 0.00180`
- `P(ตาก) =  320 / 1,000,000  = 0.00032`
- `P(ลม)  = 3600 / 1,000,000  = 0.00360`

**Candidate A** — ตา | กลม:

$$P(A) = 0.00420 \times 0.00180 = 7.5600 \times 10^{-6} \qquad (\log_{10} = -5.1215)$$

**Candidate B** — ตาก | ลม:

$$P(B) = 0.00032 \times 0.00360 = 1.1520 \times 10^{-6} \qquad (\log_{10} = -5.9385)$$

$$\frac{P(A)}{P(B)} = \frac{7.5600\times10^{-6}}{1.1520\times10^{-6}} = \mathbf{6.5625}$$

**The unigram model prefers ตา | กลม ("round eyes") by 6.56×** — it agrees with BMM and overrules FMM, driven entirely by the fact that ตาก (320) is far rarer than ตา (4200).

Notice what we gained: instead of an arbitrary scanning direction, we now have a *number* we can argue with — and a knob we can improve with better data.

## 1.7 a bigram model

The unigram model has no notion of context, so it will *always* choose ตา|กลม, in every sentence. But consider the phrase **นั่งตากลม** ("sitting, airing in the wind") — here the intended reading is clearly ตาก|ลม.

A **bigram** model conditions each word on the previous one:

$$P(W) \approx \prod_{k=1}^{n} P(w_k \mid w_{k-1}), \qquad P(w_k \mid w_{k-1}) = \frac{C(w_{k-1}, w_k)}{C(w_{k-1})}$$

With the preceding word fixed to **นั่ง** (to sit), the two candidates become:

- A: `นั่ง ตา กลม` → `P(ตา|นั่ง) × P(กลม|ตา)`
- B: `นั่ง ตาก ลม` → `P(ตาก|นั่ง) × P(ลม|ตาก)`

In [22]:
# ---- toy bigram counts ------------------------------------------------------
UNI = {"นั่ง": 2000, "ตา": 4200, "ตาก": 320}
BI = {
    ("นั่ง", "ตา"):  4,     # 'sit eye'          - almost never happens
    ("นั่ง", "ตาก"): 180,   # 'sit airing'       - a normal collocation
    ("ตา", "กลม"):   380,   # 'eye round'        - fine, but ตา precedes many things
    ("ตาก", "ลม"):   240,   # 'air out + wind'   - ตาก is almost always followed by ลม
}

def bigram_prob(prev, word):
    return BI.get((prev, word), 0) / UNI[prev]

print("Context: the previous word is นั่ง ('to sit')\n")

p_A = bigram_prob("นั่ง", "ตา") * bigram_prob("ตา", "กลม")
p_B = bigram_prob("นั่ง", "ตาก") * bigram_prob("ตาก", "ลม")

print(f"A: P(ตา|นั่ง)  = {BI[('นั่ง','ตา')]}/{UNI['นั่ง']} = {bigram_prob('นั่ง','ตา'):.5f}"
      f"   P(กลม|ตา)  = {BI[('ตา','กลม')]}/{UNI['ตา']} = {bigram_prob('ตา','กลม'):.6f}")
print(f"   -> P(A) = {p_A:.6e}")
print(f"B: P(ตาก|นั่ง) = {BI[('นั่ง','ตาก')]}/{UNI['นั่ง']} = {bigram_prob('นั่ง','ตาก'):.5f}"
      f"   P(ลม|ตาก)  = {BI[('ตาก','ลม')]}/{UNI['ตาก']} = {bigram_prob('ตาก','ลม'):.6f}")
print(f"   -> P(B) = {p_B:.6e}")

print(f"\nWinner (bigram): {'ตาก|ลม' if p_B > p_A else 'ตา|กลม'}"
      f"   by a factor of {max(p_A,p_B)/min(p_A,p_B):.2f}x")
print("\nThe unigram model preferred ตา|กลม by 6.56x.")
print("Adding ONE word of context reverses the decision - and far more confidently.")

Context: the previous word is นั่ง ('to sit')

A: P(ตา|นั่ง)  = 4/2000 = 0.00200   P(กลม|ตา)  = 380/4200 = 0.090476
   -> P(A) = 1.809524e-04
B: P(ตาก|นั่ง) = 180/2000 = 0.09000   P(ลม|ตาก)  = 240/320 = 0.750000
   -> P(B) = 6.750000e-02

Winner (bigram): ตาก|ลม   by a factor of 373.03x

The unigram model preferred ตา|กลม by 6.56x.
Adding ONE word of context reverses the decision - and far more confidently.


### The arithmetic, written out

**Candidate A** — นั่ง | ตา | กลม:

$$P(\text{ตา} \mid \text{นั่ง}) = \frac{4}{2000} = 0.00200, \qquad P(\text{กลม} \mid \text{ตา}) = \frac{380}{4200} = 0.090476$$
$$P(A) = 0.00200 \times 0.090476 = 1.809524 \times 10^{-4}$$

**Candidate B** — นั่ง | ตาก | ลม:

$$P(\text{ตาก} \mid \text{นั่ง}) = \frac{180}{2000} = 0.09000, \qquad P(\text{ลม} \mid \text{ตาก}) = \frac{240}{320} = 0.75000$$
$$P(B) = 0.09000 \times 0.75000 = 6.750000 \times 10^{-2}$$

$$\frac{P(B)}{P(A)} = \frac{6.75 \times 10^{-2}}{1.809524 \times 10^{-4}} = \mathbf{373.03}$$

## 1.8 dynamic programming


In [23]:
def dp_segment(text, dictionary, max_word_len=8):
    """Best segmentation under the unigram LM, via dynamic programming."""
    n = len(text)
    NEG_INF = float("-inf")
    best = [NEG_INF] * (n + 1)
    back = [None] * (n + 1)
    best[0] = 0.0

    for i in range(1, n + 1):
        for L in range(1, min(max_word_len, i) + 1):
            word = text[i - L:i]
            if word in dictionary or L == 1:          # L==1 = unknown-char fallback
                if best[i - L] == NEG_INF:
                    continue
                score = best[i - L] + math.log10(unigram_prob(word))
                if score > best[i]:
                    best[i] = score
                    back[i] = (i - L, word)

    tokens, i = [], n
    while i > 0:
        prev, word = back[i]
        tokens.insert(0, word)
        i = prev
    return tokens, best[n]


for text in ["ตากลม", "ตากลมนอนหลับ"]:
    tokens, logp = dp_segment(text, THAI_DICT)
    print(f"{text:<16} -> {'|'.join(tokens):<28} log10 P = {logp:.4f}")

print("\nCompare the greedy algorithms on the longer string:")
print("  FMM:", "|".join(forward_max_match("ตากลมนอนหลับ", THAI_DICT)))
print("  BMM:", "|".join(backward_max_match("ตากลมนอนหลับ", THAI_DICT)))
print("  DP :", "|".join(dp_segment("ตากลมนอนหลับ", THAI_DICT)[0]))

ตากลม            -> ตา|กลม                       log10 P = -5.1215
ตากลมนอนหลับ     -> ตา|กลม|นอนหลับ               log10 P = -8.1672

Compare the greedy algorithms on the longer string:
  FMM: ตาก|ลม|นอนหลับ
  BMM: ตา|กลม|นอนหลับ
  DP : ตา|กลม|นอนหลับ


## 1.9 How segmentation is evaluated

Since the output is a set of boundaries (equivalently, a set of word spans), segmenters are scored with **precision, recall, and F1 over words** — a predicted word counts as correct only if **both** its boundaries match the gold word exactly.

$$P = \frac{|\text{correct words}|}{|\text{predicted words}|}, \quad R = \frac{|\text{correct words}|}{|\text{gold words}|}, \quad F_1 = \frac{2PR}{P+R}$$

In [24]:
def word_f1(gold_tokens, pred_tokens):
    """Span-exact P/R/F1 over words (boundaries must match exactly)."""
    def spans(tokens):
        out, pos = [], 0
        for t in tokens:
            out.append((pos, pos + len(t)))
            pos += len(t)
        return out

    gold, pred = set(spans(gold_tokens)), set(spans(pred_tokens))
    tp = len(gold & pred)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


gold = ["ตา", "กลม", "นอนหลับ"]          # suppose this is the gold answer
for name, pred in [("FMM", forward_max_match("ตากลมนอนหลับ", THAI_DICT)),
                   ("BMM", backward_max_match("ตากลมนอนหลับ", THAI_DICT)),
                   ("DP ", dp_segment("ตากลมนอนหลับ", THAI_DICT)[0])]:
    p, r, f = word_f1(gold, pred)
    print(f"{name}: {'|'.join(pred):<28} P={p:.3f} R={r:.3f} F1={f:.3f}")

FMM: ตาก|ลม|นอนหลับ               P=0.333 R=0.333 F1=0.333
BMM: ตา|กลม|นอนหลับ               P=1.000 R=1.000 F1=1.000
DP : ตา|กลม|นอนหลับ               P=1.000 R=1.000 F1=1.000


In [25]:
# ---- OPTIONAL: compare against real PyThaiNLP engines ----------------------
# Skips harmlessly if pythainlp is not installed.
try:
    from pythainlp.tokenize import word_tokenize
    from pythainlp.corpus.common import thai_words
    from pythainlp.util import Trie
    import pythainlp

    print(f"pythainlp version {pythainlp.__version__}\n")

    words = thai_words()
    print("Is each string an entry in PyThaiNLP's dictionary?")
    for w in ["ตากลม", "ตาก", "ตา", "กลม", "ลม", "เชียงใหม่"]:
        print(f"   {w:<12} {w in words}")

    print("\nSegmenting 'ตากลม' with the full dictionary:")
    for engine in ["newmm", "longest", "attacut", "deepcut"]:
        try:
            print(f"   {engine:>9}: {'|'.join(word_tokenize('ตากลม', engine=engine))}")
        except Exception as e:
            print(f"   {engine:>9}: unavailable ({type(e).__name__})")
    print("   -> 'ตากลม' is a dictionary word, so it is never split.")

    # Force the ambiguity by removing the compound from the dictionary.
    no_compound = Trie([w for w in words if w != "ตากลม"])
    print("\nSame engine, but with 'ตากลม' REMOVED from the dictionary:")
    print(f"   ตากลม        -> {'|'.join(word_tokenize('ตากลม', custom_dict=no_compound))}")
    print(f"   นั่งตากลม     -> {'|'.join(word_tokenize('นั่งตากลม', custom_dict=no_compound))}")
    print("""
   -> Forced to choose, newmm picks ตา|กลม - matching our unigram model (6.56x)
      and BMM. But the context word นั่ง does NOT flip it to ตาก|ลม, because
      maximal matching has no bigram model. That gap is exactly what section 1.7
      illustrates, and what neural segmenters (attacut/deepcut) exist to close.""")
except ImportError:
    print("pythainlp not installed - skipping this optional comparison.")
    print("(pip install pythainlp)  Our from-scratch results above stand on their own.")

pythainlp version 5.3.7

Is each string an entry in PyThaiNLP's dictionary?
   ตากลม        True
   ตาก          True
   ตา           True
   กลม          True
   ลม           True
   เชียงใหม่    True

Segmenting 'ตากลม' with the full dictionary:
       newmm: ตากลม
     longest: ตากลม
     attacut: unavailable (ModuleNotFoundError)


c:\Users\satidasoo\AppData\Local\anaconda3\envs\nlp\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\satidasoo\AppData\Local\anaconda3\envs\nlp\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
     deepcut: ตากลม
   -> 'ตากลม' is a dictionary word, so it is never split.

Same engine, but with 'ตากลม' REMOVED from the dictionary:
   ตากลม        -> ตา|กลม
   นั่งตากลม     -> นั่ง|ตา|กลม

   -> Forced to choose, newmm picks ตา|กลม - matching our unigram model (6.56x)
      and BMM. But the context word นั่ง does NOT flip it to ตาก|ลม, because
      maximal matching has no bigram model. That gap is exactly what section 1.7
      illustrates, and what neural segmenters (attacut/deepcut) exist to close.


## DIY

---
### 🔧 DIY 1 — Trace Maximum Matching yourself

The string **อากาศตากลม** ("the weather/air, airing in the wind") extends our ambiguous case with a preceding word.

1. Predict on paper what FMM and BMM will output, using `THAI_DICT`.
2. Then complete the code below to check your prediction.
3. Answer in the variables provided: does the boundary between ตาก/ตา land in the same place for both?

In [26]:
# === DIY 1 ===
text = "อากาศตากลม"

## HINT
# fmm_out = ___(___, ___, verbose=True)
# bmm_out = ___(___, ___, verbose=True)
# they_disagree = (___ != ___)
##

# TODO 1: run forward_max_match on 'text' with THAI_DICT and verbose=True
fmm_out = None      # <-- replace None

# TODO 2: run backward_max_match on 'text' with THAI_DICT and verbose=True
bmm_out = None      # <-- replace None

# TODO 3: set this to True if the two algorithms produced DIFFERENT segmentations
they_disagree = None    # <-- replace None

print("FMM:", fmm_out)
print("BMM:", bmm_out)
print("Disagree?", they_disagree)

FMM: None
BMM: None
Disagree? None


---
### 🔧 DIY 2 — Resolve an ambiguity with probabilities

The string **ลมกลม** could be segmented `ลม|กลม` ("round wind"). Suppose you add a new word to the corpus:

| word | count |
|---|---|
| ลมกลม (a hypothetical single word) | 50 |

Compute, using `unigram_prob`, whether the unigram model prefers the **one-word** analysis `["ลมกลม"]` or the **two-word** analysis `["ลม", "กลม"]`, and by what factor. Think first: a single rare word vs. two common words — which wins, and why?

In [27]:
# === DIY 2 ===
UNIGRAM_COUNTS["ลมกลม"] = 50      # register the hypothetical compound

one_word = ["ลมกลม"]
two_words = ["ลม", "กลม"]

## HINT
# p_one, log_one = ___(one_word)
# p_two, log_two = score_segmentation(___)

# TODO 1: score both analyses with score_segmentation()
p_one, log_one = None, None       # <-- replace
p_two, log_two = None, None       # <-- replace


##HINT
# winner = one_word if ____ > _____ else two_words
# factor = ___(p_one, p_two) / ___(p_one, p_two)

# TODO 2: which analysis wins, and by what factor?
winner = None                      # <-- replace with one_word or two_words
factor = None                      # <-- replace with the ratio (>1)

print(f"one word : {p_one}")
print(f"two words: {p_two}")
print(f"winner   : {winner}  by {factor}x")

one word : None
two words: None
winner   : None  by Nonex


---
# Part 2 — Part-of-Speech Tagging

## 2.1 The task

**POS tagging** assigns each word in a sequence its part of speech: `Janet/NNP will/MD back/VB the/DT bill/NN`. It is the canonical **sequence labeling** 

In [28]:
# ---- Baseline: most-frequent-tag ------------------------------------------
# A tiny hand-tagged "corpus" (word, tag) pairs.
TAGGED_CORPUS = [
    ("Secretariat", "NNP"), ("is", "VBZ"), ("expected", "VBN"), ("to", "TO"),
    ("race", "VB"), ("tomorrow", "NN"),
    ("the", "DT"), ("race", "NN"), ("for", "IN"), ("outer", "JJ"), ("space", "NN"),
    ("the", "DT"), ("race", "NN"), ("was", "VBD"), ("close", "JJ"),
    ("people", "NNS"), ("continue", "VBP"), ("to", "TO"), ("inquire", "VB"),
    ("the", "DT"), ("reason", "NN"), ("for", "IN"), ("the", "DT"), ("race", "NN"),
]

from collections import Counter, defaultdict

tag_counts = defaultdict(Counter)
for word, tag in TAGGED_CORPUS:
    tag_counts[word.lower()][tag] += 1

def most_frequent_tag(word, default="NN"):
    counts = tag_counts.get(word.lower())
    return counts.most_common(1)[0][0] if counts else default

print("Learned tag distribution for 'race':", dict(tag_counts["race"]))
p_nn = tag_counts["race"]["NN"] / sum(tag_counts["race"].values())
p_vb = tag_counts["race"]["VB"] / sum(tag_counts["race"].values())
print(f"  P(NN|race) = {p_nn:.2f}      P(VB|race) = {p_vb:.2f}")
print(f"  -> most-frequent-tag baseline always says: {most_frequent_tag('race')}\n")

sentence = ["Secretariat", "is", "expected", "to", "race", "tomorrow"]
gold      = ["NNP", "VBZ", "VBN", "TO", "VB", "NN"]
baseline  = [most_frequent_tag(w) for w in sentence]

print(f"{'word':<14}{'baseline':<10}{'gold':<8}")
for w, b, g in zip(sentence, baseline, gold):
    print(f"{w:<14}{b:<10}{g:<8}{'' if b == g else '  <-- ERROR'}")
errors = sum(b != g for b, g in zip(baseline, gold))
print(f"\nBaseline errors: {errors}/{len(gold)}")

Learned tag distribution for 'race': {'VB': 1, 'NN': 3}
  P(NN|race) = 0.75      P(VB|race) = 0.25
  -> most-frequent-tag baseline always says: NN

word          baseline  gold    
Secretariat   NNP       NNP     
is            VBZ       VBZ     
expected      VBN       VBN     
to            TO        TO      
race          NN        VB        <-- ERROR
tomorrow      NN        NN      

Baseline errors: 1/6


## 2.2 Stochastic tagging: the HMM formalism

Instead of rules, model the sequence probabilistically. We want the most probable **tag sequence** given the **word sequence**:

$$\hat{t}_{1:n} = \arg\max_{t_{1:n}} P(t_{1:n} \mid w_{1:n})$$

Apply Bayes' rule and drop the constant denominator `P(w)`:

$$\hat{t}_{1:n} = \arg\max_{t_{1:n}} \frac{P(w_{1:n} \mid t_{1:n})\, P(t_{1:n})}{P(w_{1:n})} = \arg\max_{t_{1:n}} \underbrace{P(w_{1:n} \mid t_{1:n})}_{\text{likelihood}} \; \underbrace{P(t_{1:n})}_{\text{prior}}$$

This is still intractable, so an HMM makes **two simplifying assumptions**:

1. **Output independence** — a word depends only on its own tag, not on neighbouring words or tags:
   $$P(w_{1:n} \mid t_{1:n}) \approx \prod_{i=1}^{n} P(w_i \mid t_i)$$
2. **Markov assumption (bigram)** — a tag depends only on the previous tag:
   $$P(t_{1:n}) \approx \prod_{i=1}^{n} P(t_i \mid t_{i-1})$$

Together:

$$\hat{t}_{1:n} = \arg\max_{t_{1:n}} \prod_{i=1}^{n} \underbrace{P(w_i \mid t_i)}_{\textbf{B} \text{ emission}} \cdot \underbrace{P(t_i \mid t_{i-1})}_{\textbf{A} \text{ transition}}$$

### The formal components

An HMM is the tuple $(Q, A, B, \pi)$:

| Symbol | Name | In POS tagging |
|---|---|---|
| $Q = q_1 \ldots q_N$ | **states** | the $N$ tags (here $N=7$) |
| $A = [a_{ij}]$ | **transition matrix** | $a_{ij} = P(t_j \mid t_i)$, learned as $C(t_i, t_j)/C(t_i)$ |
| $B = [b_j(o_t)]$ | **emission / observation likelihoods** | $b_j(o_t) = P(w_t \mid t_j)$, learned as $C(t_j, w_t)/C(t_j)$ |
| $\pi$ (or the `<s>` row of $A$) | **initial distribution** | probability each tag starts a sentence |
| $O = o_1 \ldots o_T$ | **observations** | the $T$ words to be tagged |

The states are "hidden" because we observe the words, not the tags. Training an HMM tagger on a tagged corpus is just **counting** (as in the $4046/13124$ example above) — the hard part is **decoding**: finding the best tag sequence out of exponentially many. That is Viterbi's job.

## 2.3 The textbook matrices - Janet Example

In [29]:
# ---- The HMM: J&M Ch.18 Fig 18.12 (A) and Fig 18.13 (B), verbatim ---------
TAGS = ["NNP", "MD", "VB", "JJ", "NN", "RB", "DT"]

A = {
    "<s>": {"NNP": 0.2767, "MD": 0.0006, "VB": 0.0031, "JJ": 0.0453, "NN": 0.0449, "RB": 0.0510, "DT": 0.2026},
    "NNP": {"NNP": 0.3777, "MD": 0.0110, "VB": 0.0009, "JJ": 0.0084, "NN": 0.0584, "RB": 0.0090, "DT": 0.0025},
    "MD":  {"NNP": 0.0008, "MD": 0.0002, "VB": 0.7968, "JJ": 0.0005, "NN": 0.0008, "RB": 0.1698, "DT": 0.0041},
    "VB":  {"NNP": 0.0322, "MD": 0.0005, "VB": 0.0050, "JJ": 0.0837, "NN": 0.0615, "RB": 0.0514, "DT": 0.2231},
    "JJ":  {"NNP": 0.0366, "MD": 0.0004, "VB": 0.0001, "JJ": 0.0733, "NN": 0.4509, "RB": 0.0036, "DT": 0.0036},
    "NN":  {"NNP": 0.0096, "MD": 0.0176, "VB": 0.0014, "JJ": 0.0086, "NN": 0.1216, "RB": 0.0177, "DT": 0.0068},
    "RB":  {"NNP": 0.0068, "MD": 0.0102, "VB": 0.1011, "JJ": 0.1012, "NN": 0.0120, "RB": 0.0728, "DT": 0.0479},
    "DT":  {"NNP": 0.1147, "MD": 0.0021, "VB": 0.0002, "JJ": 0.2157, "NN": 0.4744, "RB": 0.0102, "DT": 0.0017},
}

B = {
    "NNP": {"Janet": 0.000032, "will": 0.0,      "back": 0.0,      "the": 0.000048, "bill": 0.0},
    "MD":  {"Janet": 0.0,      "will": 0.308431, "back": 0.0,      "the": 0.0,      "bill": 0.0},
    "VB":  {"Janet": 0.0,      "will": 0.000028, "back": 0.000672, "the": 0.0,      "bill": 0.000028},
    "JJ":  {"Janet": 0.0,      "will": 0.0,      "back": 0.000340, "the": 0.0,      "bill": 0.0},
    "NN":  {"Janet": 0.0,      "will": 0.000200, "back": 0.000223, "the": 0.0,      "bill": 0.002337},
    "RB":  {"Janet": 0.0,      "will": 0.0,      "back": 0.010446, "the": 0.0,      "bill": 0.0},
    "DT":  {"Janet": 0.0,      "will": 0.0,      "back": 0.0,      "the": 0.506099, "bill": 0.0},
}

SENTENCE = ["Janet", "will", "back", "the", "bill"]
GOLD_TAGS = ["NNP", "MD", "VB", "DT", "NN"]      # J&M eq. 18.20

# quick structural checks
assert len(TAGS) == 7 and len(SENTENCE) == 5
assert A["MD"]["VB"] == 0.7968 and B["DT"]["the"] == 0.506099
print(f"HMM ready: N = {len(TAGS)} tags, T = {len(SENTENCE)} words")
print(f"P(VB|MD)   = {A['MD']['VB']}      <- after a modal, expect a base-form verb")
print(f"P(NN|DT)   = {A['DT']['NN']}      <- after a determiner, expect a noun")
print(f"P(the|DT)  = {B['DT']['the']}")

HMM ready: N = 7 tags, T = 5 words
P(VB|MD)   = 0.7968      <- after a modal, expect a base-form verb
P(NN|DT)   = 0.4744      <- after a determiner, expect a noun
P(the|DT)  = 0.506099


## 2.4 The Viterbi algorithm

Viterbi is dynamic programming over a **trellis** of $T \times N$ cells (one per word × tag). Each cell stores the probability of the single best path that ends in that cell, plus a backpointer.

**Definition.** $v_t(j)$ = the probability of the most likely tag sequence for the first $t$ words that ends in tag $j$.

**Initialisation** ($t = 1$):
$$v_1(j) = P(t_j \mid \texttt{<s>}) \cdot b_j(o_1) = a_{\texttt{<s>},j} \cdot b_j(o_1)$$

**Recursion** ($t > 1$):
$$v_t(j) = \max_{i=1}^{N} \Big[\, v_{t-1}(i) \cdot a_{ij} \cdot b_j(o_t) \,\Big]$$
$$bt_t(j) = \arg\max_{i=1}^{N} \Big[\, v_{t-1}(i) \cdot a_{ij} \cdot b_j(o_t) \,\Big]$$

**Termination:** best final score $= \max_j v_T(j)$; then follow backpointers from $\arg\max_j v_T(j)$ back to $t=1$.

The crucial insight is the **max, not sum**: because each cell only needs the *best* way to reach it, all the paths that pass through a cell can be summarised by one number — so we never enumerate paths.

In [30]:
def viterbi(words, tags, A, B, start="<s>", trace=False):
    """Classic Viterbi decoding. Returns (best_path, best_prob, v, backpointer, n_ops)."""
    T = len(words)
    v = [dict() for _ in range(T)]
    bp = [dict() for _ in range(T)]
    n_ops = 0                                    # count score computations

    # --- initialisation (t = 1) ---
    for tag in tags:
        v[0][tag] = A[start][tag] * B[tag].get(words[0], 0.0)
        bp[0][tag] = start
        n_ops += 1

    # --- recursion (t = 2..T) ---
    for t in range(1, T):
        for tag in tags:
            best_score, best_prev = -1.0, None
            for prev in tags:
                score = v[t - 1][prev] * A[prev][tag] * B[tag].get(words[t], 0.0)
                n_ops += 1
                if score > best_score:
                    best_score, best_prev = score, prev
            v[t][tag] = best_score
            bp[t][tag] = best_prev

    # --- termination + backtrace ---
    best_final = max(tags, key=lambda tag: v[T - 1][tag])
    best_prob = v[T - 1][best_final]
    path = [best_final]
    for t in range(T - 1, 0, -1):
        path.insert(0, bp[t][path[0]])

    if trace:
        for t, word in enumerate(words):
            print(f"--- t={t+1}  word = {word!r}")
            for tag in tags:
                val = v[t][tag]
                if val > 0:
                    src = f"   <- from {bp[t][tag]}"
                    print(f"      v{t+1}({tag:<3}) = {val:.6e}{src}")
                else:
                    print(f"      v{t+1}({tag:<3}) = 0")
        print()
    return path, best_prob, v, bp, n_ops


path, best_prob, v, bp, n_ops = viterbi(SENTENCE, TAGS, A, B, trace=True)

print("=" * 66)
print("best path       :", " -> ".join(path))
print("best probability:", f"{best_prob:.6e}")
print("gold (J&M 18.20):", " -> ".join(GOLD_TAGS))
print("score computations performed:", n_ops)
print("=" * 66)

assert path == GOLD_TAGS, f"got {path}"
print("\n[ok] Viterbi reproduces the textbook's gold sequence NNP MD VB DT NN.")

--- t=1  word = 'Janet'
      v1(NNP) = 8.854400e-06   <- from <s>
      v1(MD ) = 0
      v1(VB ) = 0
      v1(JJ ) = 0
      v1(NN ) = 0
      v1(RB ) = 0
      v1(DT ) = 0
--- t=2  word = 'will'
      v2(NNP) = 0
      v2(MD ) = 3.004069e-08   <- from NNP
      v2(VB ) = 2.231309e-13   <- from NNP
      v2(JJ ) = 0
      v2(NN ) = 1.034194e-10   <- from NNP
      v2(RB ) = 0
      v2(DT ) = 0
--- t=3  word = 'back'
      v3(NNP) = 0
      v3(MD ) = 0
      v3(VB ) = 1.608527e-11   <- from MD
      v3(JJ ) = 5.106917e-15   <- from MD
      v3(NN ) = 5.359258e-15   <- from MD
      v3(RB ) = 5.328409e-11   <- from MD
      v3(DT ) = 0
--- t=4  word = 'the'
      v4(NNP) = 2.486140e-17   <- from VB
      v4(MD ) = 0
      v4(VB ) = 0
      v4(JJ ) = 0
      v4(NN ) = 0
      v4(RB ) = 0
      v4(DT ) = 1.816199e-12   <- from VB
--- t=5  word = 'bill'
      v5(NNP) = 0
      v5(MD ) = 0
      v5(VB ) = 1.017072e-20   <- from DT
      v5(JJ ) = 0
      v5(NN ) = 2.013571e-15   <- from DT

## 2.5 Practical matter: log-space Viterbi

Look at the final probability: $2.01 \times 10^{-15}$ for a 5-word sentence. For a 30-word sentence the numbers **underflow to 0.0** in floating point, and every path ties at zero.

The fix is standard: work with $\log$ probabilities, so products become sums.

$$v_t(j) = \max_i \big[\, v_{t-1}(i) + \log a_{ij} + \log b_j(o_t) \,\big]$$

Sums of logs are numerically stable, and $\arg\max$ is unchanged because $\log$ is monotonically increasing. Every production decoder does this.

In [31]:
NEG_INF = float("-inf")

def log_(x):
    return math.log(x) if x > 0 else NEG_INF

def viterbi_log(words, tags, A, B, start="<s>"):
    """Viterbi in log space - what you should actually ship."""
    T = len(words)
    v = [dict() for _ in range(T)]
    bp = [dict() for _ in range(T)]

    for tag in tags:
        v[0][tag] = log_(A[start][tag]) + log_(B[tag].get(words[0], 0.0))
        bp[0][tag] = start

    for t in range(1, T):
        for tag in tags:
            best_score, best_prev = NEG_INF, None
            for prev in tags:
                score = v[t - 1][prev] + log_(A[prev][tag]) + log_(B[tag].get(words[t], 0.0))
                if score > best_score:
                    best_score, best_prev = score, prev
            v[t][tag], bp[t][tag] = best_score, best_prev

    best_final = max(tags, key=lambda tag: v[T - 1][tag])
    path = [best_final]
    for t in range(T - 1, 0, -1):
        path.insert(0, bp[t][path[0]])
    return path, v[T - 1][best_final]


log_path, log_score = viterbi_log(SENTENCE, TAGS, A, B)
print("log-space path :", " -> ".join(log_path))
print(f"log probability: {log_score:.6f}")
print(f"exp(log prob)  : {math.exp(log_score):.6e}")
print(f"linear Viterbi : {best_prob:.6e}")
assert log_path == path
print("\n[ok] Same path, no underflow risk.")

print("\nUnderflow demo - repeat the sentence 8 times (40 words):")
long_sentence = SENTENCE * 8
lin_p = 1.0
for i, w in enumerate(long_sentence):
    lin_p *= 0.001            # a stand-in per-word factor
print(f"  a linear probability of this magnitude: {lin_p:.3e}"
      f"  -> underflows to {0.0 if lin_p == 0 else lin_p} at ~10^-308")
print(f"  the same value in log space:            {40 * math.log(0.001):.3f}   (perfectly stable)")

log-space path : NNP -> MD -> VB -> DT -> NN
log probability: -33.838867
exp(log prob)  : 2.013571e-15
linear Viterbi : 2.013571e-15

[ok] Same path, no underflow risk.

Underflow demo - repeat the sentence 8 times (40 words):
  a linear probability of this magnitude: 1.000e-120  -> underflows to 1.0000000000000012e-120 at ~10^-308
  the same value in log space:            -276.310   (perfectly stable)


## 2.6 Conditional Random Fields (CRFs)

A **linear-chain CRF** is the discriminative answer. It models $P(\text{tags} \mid \text{words})$ **directly**, so it can use arbitrary, overlapping, non-independent features of the *whole* input:

$$P(Y \mid X) = \frac{1}{Z(X)} \exp\left( \sum_{k=1}^{K} w_k \sum_{i=1}^{n} f_k(y_{i-1}, y_i, X, i) \right)$$

where

- $f_k(y_{i-1}, y_i, X, i)$ is **feature function** $k$ evaluated at position $i$ — it may look at the previous tag, the current tag, and *any* part of the input $X$;
- $w_k$ is the learned weight for that feature;
- $Z(X)$ is the normalisation constant (**partition function**) summing over all possible tag sequences — this is what makes the CRF **globally normalised** rather than making a locally normalised decision per position.

In [32]:
def word_shape(word):
    """DC10-30 -> XXdd-dd"""
    out = []
    for ch in word:
        if ch.isupper():   out.append("X")
        elif ch.islower(): out.append("x")
        elif ch.isdigit(): out.append("d")
        else:              out.append(ch)
    return "".join(out)


def short_shape(word):
    """DC10-30 -> Xd-d  (collapse consecutive identical shape symbols)"""
    shape = word_shape(word)
    out = []
    for ch in shape:
        if not out or out[-1] != ch:
            out.append(ch)
    return "".join(out)


CITY_GAZETTEER = {"Chicago", "Bangkok", "Denver", "Dallas", "San Francisco"}


def crf_features(sentence, i, pos_tags=None):
    """Feature dict for position i, following the textbook's templates."""
    w = sentence[i]
    feats = {
        "bias": 1.0,
        "w[0]": w,
        "w[0].lower": w.lower(),
        "w[0].isupper": w.isupper(),
        "w[0].istitle": w.istitle(),
        "w[0].isdigit": w.isdigit(),
        "shape": word_shape(w),
        "short_shape": short_shape(w),
        "in_gazetteer": w in CITY_GAZETTEER,
    }
    for n in range(1, 5):                       # prefixes/suffixes up to length 4
        if len(w) >= n:
            feats[f"prefix[:{n}]"] = w[:n]
            feats[f"suffix[-{n}:]"] = w[-n:]
    if i > 0:
        feats["w[-1]"] = sentence[i - 1]
        feats["w[-1].istitle"] = sentence[i - 1].istitle()
        feats["shape[-1]"] = word_shape(sentence[i - 1])
    else:
        feats["BOS"] = True                     # beginning of sentence
    if i < len(sentence) - 1:
        feats["w[+1]"] = sentence[i + 1]
        feats["shape[+1]"] = word_shape(sentence[i + 1])
    else:
        feats["EOS"] = True
    if pos_tags:
        feats["pos[0]"] = pos_tags[i]
        if i > 0:
            feats["pos[-1]"] = pos_tags[i - 1]
    return feats


print("Word shapes:")
for w in ["DC10-30", "Villanueva", "IBM", "iPhone", "2026", "Bangkok"]:
    print(f"  {w:<12} shape = {word_shape(w):<12} short = {short_shape(w)}")

sent = ["Jane", "Villanueva", "of", "United", "Airlines", "Holding", "discussed",
        "the", "Chicago", "route", "."]

print("\nFeatures for the UNKNOWN word 'Villanueva' (i=1):")
for k, val in crf_features(sent, 1).items():
    print(f"  {k:<18} = {val}")

print("\nWhy this works: the model has never seen 'Villanueva', but")
print("  shape=Xxxxxxxxxx + istitle + previous word 'Jane' + suffix '-eva'")
print("is strong evidence for I-PER. An HMM could only ask P(Villanueva | I-PER) ~ 0.")

Word shapes:
  DC10-30      shape = XXdd-dd      short = Xd-d
  Villanueva   shape = Xxxxxxxxxx   short = Xx
  IBM          shape = XXX          short = X
  iPhone       shape = xXxxxx       short = xXx
  2026         shape = dddd         short = d
  Bangkok      shape = Xxxxxxx      short = Xx

Features for the UNKNOWN word 'Villanueva' (i=1):
  bias               = 1.0
  w[0]               = Villanueva
  w[0].lower         = villanueva
  w[0].isupper       = False
  w[0].istitle       = True
  w[0].isdigit       = False
  shape              = Xxxxxxxxxx
  short_shape        = Xx
  in_gazetteer       = False
  prefix[:1]         = V
  suffix[-1:]        = a
  prefix[:2]         = Vi
  suffix[-2:]        = va
  prefix[:3]         = Vil
  suffix[-3:]        = eva
  prefix[:4]         = Vill
  suffix[-4:]        = ueva
  w[-1]              = Jane
  w[-1].istitle      = True
  shape[-1]          = Xxxx
  w[+1]              = of
  shape[+1]          = xx

Why this works: the model has neve

## DIY

---
### 🔧 DIY 3 — Compute a Viterbi cell by hand

Compute $v_2(\text{NN})$ — the best path probability for tagging *will* as a common noun — **on paper first**, then verify.

You need: $v_1(\text{NNP}) = 8.8544 \times 10^{-6}$, $a_{\text{NNP},\text{NN}}$ from Figure 18.12, and $b_{\text{NN}}(\text{will})$ from Figure 18.13. (Every other predecessor is 0, so the max has exactly one non-zero candidate.)

Then answer: how many times larger is $v_2(\text{MD})$ than $v_2(\text{NN})$, and what does that ratio *mean* linguistically?

In [33]:
# === DIY 3 ===
v1_nnp = A["<s>"]["NNP"] * B["NNP"]["Janet"]     # given: 8.8544e-6


## HINT
# a_nnp_nn = A["___"]["___"]       # 0.0584 
# b_nn_will = B["___"]["___"]      # 0.000200 
# TODO 1: look up the two probabilities you need
a_nnp_nn = None      # <-- P(NN | NNP) from Figure 18.12
b_nn_will = None     # <-- P(will | NN) from Figure 18.13


## HINT
# v2_nn = v1_nnp * a_nnp_nn * b_nn_will
# v2_md = v1_nnp * A["NNP"]["MD"] * B["MD"]["will"]
# ratio = v2_md / v2_nn

# TODO 2: compute v2(NN)
v2_nn = None         # <-- replace

# TODO 3: how many times larger is v2(MD) than v2(NN)?
v2_md = v1_nnp * A["NNP"]["MD"] * B["MD"]["will"]
ratio = None         # <-- replace

print(f"v2(NN) = {v2_nn}")
print(f"v2(MD) = {v2_md:.6e}")
print(f"MD is {ratio} times more likely than NN")

v2(NN) = None
v2(MD) = 3.004069e-08
MD is None times more likely than NN


---
# Part 3 — Sequence Labeling and Named Entity Recognition

In [34]:
# ---- BIO utilities --------------------------------------------------------
def spans_to_bio(tokens, spans):
    """spans: list of (start, end_inclusive, type) -> BIO tag list."""
    tags = ["O"] * len(tokens)
    for start, end, etype in spans:
        tags[start] = f"B-{etype}"
        for i in range(start + 1, end + 1):
            tags[i] = f"I-{etype}"
    return tags


def bio_to_spans(tags):
    """BIO tag list -> list of (start, end_inclusive, type). Inverse of the above."""
    spans, start, etype = [], None, None
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = i, tag[2:]
        elif tag.startswith("I-") and start is not None and tag[2:] == etype:
            continue                              # still inside the same entity
        else:                                     # O, or an inconsistent I-
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = None, None
    if start is not None:
        spans.append((start, len(tags) - 1, etype))
    return spans


def bio_to_io(tags):
    return [t.replace("B-", "I-") for t in tags]


def bio_to_bioes(tags):
    """B-X I-X I-X -> B-X I-X E-X ;  a lone B-X -> S-X"""
    out = []
    spans = bio_to_spans(tags)
    out = ["O"] * len(tags)
    for start, end, etype in spans:
        if start == end:
            out[start] = f"S-{etype}"
        else:
            out[start] = f"B-{etype}"
            for i in range(start + 1, end):
                out[i] = f"I-{etype}"
            out[end] = f"E-{etype}"
    return out


# Reproduce J&M Figure 18.7
tokens = ["Jane", "Villanueva", "of", "United", "Airlines", "Holding",
          "discussed", "the", "Chicago", "route", "."]
spans = [(0, 1, "PER"), (3, 5, "ORG"), (8, 8, "LOC")]

bio = spans_to_bio(tokens, spans)
io = bio_to_io(bio)
bioes = bio_to_bioes(bio)

print(f"{'Words':<12}{'IO Label':<12}{'BIO Label':<12}{'BIOES Label':<12}")
print("-" * 48)
for tok, a, b, c in zip(tokens, io, bio, bioes):
    print(f"{tok:<12}{a:<12}{b:<12}{c:<12}")

# round-trip check: BIO -> spans -> BIO
assert bio_to_spans(bio) == spans
print(f"\n[ok] BIO round-trips to the original spans: {bio_to_spans(bio)}")

n_types = 3      # PER, ORG, LOC
print(f"\nTag inventory for {n_types} entity types: 2n+1 = {2 * n_types + 1} tags")
print(f"  {sorted(set(f'{p}-{t}' for p in 'BI' for t in ['PER','ORG','LOC']) | {'O'})}")
print("\nWhy IO is weaker: it cannot separate two adjacent same-type entities.")
adjacent = ["Bangkok", "Chiang", "Mai"]                     # two LOCs side by side
adj_bio = spans_to_bio(adjacent, [(0, 0, "LOC"), (1, 2, "LOC")])
print(f"  BIO {adj_bio} -> spans {bio_to_spans(adj_bio)}  (2 entities, correct)")
print(f"  IO  {bio_to_io(adj_bio)} -> indistinguishable from ONE 3-token LOC")

Words       IO Label    BIO Label   BIOES Label 
------------------------------------------------
Jane        I-PER       B-PER       B-PER       
Villanueva  I-PER       I-PER       E-PER       
of          O           O           O           
United      I-ORG       B-ORG       B-ORG       
Airlines    I-ORG       I-ORG       I-ORG       
Holding     I-ORG       I-ORG       E-ORG       
discussed   O           O           O           
the         O           O           O           
Chicago     I-LOC       B-LOC       S-LOC       
route       O           O           O           
.           O           O           O           

[ok] BIO round-trips to the original spans: [(0, 1, 'PER'), (3, 5, 'ORG'), (8, 8, 'LOC')]

Tag inventory for 3 entity types: 2n+1 = 7 tags
  ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']

Why IO is weaker: it cannot separate two adjacent same-type entities.
  BIO ['B-LOC', 'B-LOC', 'I-LOC'] -> spans [(0, 0, 'LOC'), (1, 2, 'LOC')]  (2 entities, co

## 3.4 Thai NER

> **นายสมชายเดินทางไปเชียงใหม่เมื่อวานนี้**

| # | Token | Gloss | BIO tag |
|---|---|---|---|
| 1 | นาย | Mr. (title) | `O` |
| 2 | สมชาย | Somchai | `B-PER` |
| 3 | เดินทาง | travel | `O` |
| 4 | ไป | go | `O` |
| 5 | เชียงใหม่ | Chiang Mai | `B-LOC` |
| 6 | เมื่อวาน | yesterday | `B-DATE` |
| 7 | นี้ | this | `I-DATE` |

Now suppose the segmenter makes **one** plausible mistake. **เชียงใหม่** ("Chiang Mai") is a compound of เชียง + ใหม่, and ใหม่ ("new") is an extremely common standalone word — precisely the situation where greedy matching or a weak model splits it:

> เชียงใหม่ → **เชียง | ใหม่**


In [35]:
# ---- correct segmentation ------------------------------------------------
gold_tokens = ["นาย", "สมชาย", "เดินทาง", "ไป", "เชียงใหม่", "เมื่อวาน", "นี้"]
gold_tags   = ["O",   "B-PER", "O",       "O",  "B-LOC",     "B-DATE",  "I-DATE"]

# ---- one plausible segmentation error: เชียงใหม่ -> เชียง | ใหม่ ----------
bad_tokens = ["นาย", "สมชาย", "เดินทาง", "ไป", "เชียง", "ใหม่", "เมื่อวาน", "นี้"]
bad_tags   = ["O",   "B-PER", "O",       "O",  "B-LOC", "O",   "B-DATE",  "I-DATE"]


def entities(tokens, tags):
    """-> set of 'surface/TYPE' strings (span-exact, surface-form comparison)."""
    return {"".join(tokens[s:e + 1]) + "/" + t for s, e, t in bio_to_spans(tags)}


gold_ents = entities(gold_tokens, gold_tags)
pred_ents = entities(bad_tokens, bad_tags)

print("Correct segmentation (7 tokens):")
for tok, tag in zip(gold_tokens, gold_tags):
    print(f"   {tok:<10} {tag}")
print(f"\n   gold entities: {sorted(gold_ents)}")

print("\nMis-segmented input (8 tokens - เชียงใหม่ was split):")
for tok, tag in zip(bad_tokens, bad_tags):
    flag = "   <-- segmentation error" if tok in ("เชียง", "ใหม่") else ""
    print(f"   {tok:<10} {tag}{flag}")
print(f"\n   predicted entities: {sorted(pred_ents)}")



Correct segmentation (7 tokens):
   นาย        O
   สมชาย      B-PER
   เดินทาง    O
   ไป         O
   เชียงใหม่  B-LOC
   เมื่อวาน   B-DATE
   นี้        I-DATE

   gold entities: ['สมชาย/PER', 'เชียงใหม่/LOC', 'เมื่อวานนี้/DATE']

Mis-segmented input (8 tokens - เชียงใหม่ was split):
   นาย        O
   สมชาย      B-PER
   เดินทาง    O
   ไป         O
   เชียง      B-LOC   <-- segmentation error
   ใหม่       O   <-- segmentation error
   เมื่อวาน   B-DATE
   นี้        I-DATE

   predicted entities: ['สมชาย/PER', 'เชียง/LOC', 'เมื่อวานนี้/DATE']


In [36]:
tp = gold_ents & pred_ents
fp = pred_ents - gold_ents
fn = gold_ents - pred_ents
p = len(tp) / len(pred_ents)
r = len(tp) / len(gold_ents)
f1 = 2 * p * r / (p + r)

print(f"\n   TP = {len(tp)} {sorted(tp)}")
print(f"   FP = {len(fp)} {sorted(fp)}   <- spurious entity created by the split")
print(f"   FN = {len(fn)} {sorted(fn)}   <- true entity destroyed by the split")
print(f"\n   Precision = {len(tp)}/{len(pred_ents)} = {p:.4f}")
print(f"   Recall    = {len(tp)}/{len(gold_ents)} = {r:.4f}")
print(f"   F1        = {f1:.4f}")
print(f"\nOne tokenization error cost {(1 - f1) * 100:.1f} F1 points on a 3-entity sentence.")



   TP = 2 ['สมชาย/PER', 'เมื่อวานนี้/DATE']
   FP = 1 ['เชียง/LOC']   <- spurious entity created by the split
   FN = 1 ['เชียงใหม่/LOC']   <- true entity destroyed by the split

   Precision = 2/3 = 0.6667
   Recall    = 2/3 = 0.6667
   F1        = 0.6667

One tokenization error cost 33.3 F1 points on a 3-entity sentence.


## 3.5 Evaluating NER: 
uses **recall, precision, and F-measure over entities**, where a predicted entity counts as correct only if **both its boundaries and its type** match the gold entity exactly:

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} = \frac{\#\text{correct entities predicted}}{\#\text{entities predicted}}$$

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\#\text{correct entities predicted}}{\#\text{entities in the gold data}}$$

$$F_1 = \frac{2 \cdot P \cdot R}{P + R} \qquad \text{(the harmonic mean of P and R)}$$

In [37]:
# ---- worked P/R/F1 on the Jane Villanueva sentence -----------------------
tokens = ["Jane", "Villanueva", "of", "United", "Airlines", "Holding",
          "discussed", "the", "Chicago", "route", "."]

gold_tags = ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "B-LOC", "O", "O"]
# A plausible system output with three different error types:
pred_tags = ["B-PER", "I-PER", "O", "O",     "B-ORG", "I-ORG", "O", "O", "B-ORG", "B-LOC", "O"]
#                                    ^boundary error        ^type error   ^spurious entity

def named_entities(tokens, tags):
    return {(" ".join(tokens[s:e + 1]), t) for s, e, t in bio_to_spans(tags)}

G = named_entities(tokens, gold_tags)
P_ = named_entities(tokens, pred_tags)

print(f"{'token':<12}{'gold':<10}{'pred':<10}")
print("-" * 32)
for tok, g, pr in zip(tokens, gold_tags, pred_tags):
    print(f"{tok:<12}{g:<10}{pr:<10}{'' if g == pr else '  <-- differs'}")

print(f"\ngold entities      ({len(G)}): {sorted(G)}")
print(f"predicted entities ({len(P_)}): {sorted(P_)}")

token       gold      pred      
--------------------------------
Jane        B-PER     B-PER     
Villanueva  I-PER     I-PER     
of          O         O         
United      B-ORG     O           <-- differs
Airlines    I-ORG     B-ORG       <-- differs
Holding     I-ORG     I-ORG     
discussed   O         O         
the         O         O         
Chicago     B-LOC     B-ORG       <-- differs
route       O         B-LOC       <-- differs
.           O         O         

gold entities      (3): [('Chicago', 'LOC'), ('Jane Villanueva', 'PER'), ('United Airlines Holding', 'ORG')]
predicted entities (4): [('Airlines Holding', 'ORG'), ('Chicago', 'ORG'), ('Jane Villanueva', 'PER'), ('route', 'LOC')]


In [38]:
tp, fp, fn = G & P_, P_ - G, G - P_
print(f"\nTP = {len(tp)}: {sorted(tp)}")
print(f"FP = {len(fp)}: {sorted(fp)}")
print(f"FN = {len(fn)}: {sorted(fn)}")

precision = len(tp) / len(P_)
recall = len(tp) / len(G)
f1 = 2 * precision * recall / (precision + recall)

print(f"\n--- entity-level (the metric that counts) ---")
print(f"Precision = TP/(TP+FP) = {len(tp)}/{len(P_)} = {precision:.4f}")
print(f"Recall    = TP/(TP+FN) = {len(tp)}/{len(G)} = {recall:.4f}")
print(f"F1        = 2PR/(P+R)  = {f1:.4f}")

correct_tokens = sum(g == pr for g, pr in zip(gold_tags, pred_tags))
token_acc = correct_tokens / len(gold_tags)
all_o_acc = gold_tags.count("O") / len(gold_tags)

print(f"\n--- token-level (the metric that lies) ---")
print(f"Token accuracy          = {correct_tokens}/{len(gold_tags)} = {token_acc:.4f}")
print(f"'predict O everywhere'  = {gold_tags.count('O')}/{len(gold_tags)} = {all_o_acc:.4f}  (with F1 = 0.0)")
print(f"\nAccuracy {token_acc:.0%}, F1 {f1:.1%}")


TP = 1: [('Jane Villanueva', 'PER')]
FP = 3: [('Airlines Holding', 'ORG'), ('Chicago', 'ORG'), ('route', 'LOC')]
FN = 2: [('Chicago', 'LOC'), ('United Airlines Holding', 'ORG')]

--- entity-level (the metric that counts) ---
Precision = TP/(TP+FP) = 1/4 = 0.2500
Recall    = TP/(TP+FN) = 1/3 = 0.3333
F1        = 2PR/(P+R)  = 0.2857

--- token-level (the metric that lies) ---
Token accuracy          = 7/11 = 0.6364
'predict O everywhere'  = 5/11 = 0.4545  (with F1 = 0.0)

Accuracy 64%, F1 28.6%


## DIY

---
### 🔧 DIY 4 — BIO-tag a Thai sentence

Take the sentence **"บริษัทปูนซิเมนต์ไทยตั้งอยู่ที่กรุงเทพมหานคร"** ("The Siam Cement Company is located in Bangkok"), pre-segmented for you below.

Gold entities: **ปูนซิเมนต์ไทย** (tokens 1–2, `ORG`) and **กรุงเทพมหานคร** (token 5, `LOC`).

1. Write the BIO tags.
2. Convert them to BIOES.
3. Verify with `bio_to_spans` that you can recover the entities.

In [39]:
# === DIY 6 ===
th_tokens = ["บริษัท", "ปูนซิเมนต์", "ไทย", "ตั้งอยู่", "ที่", "กรุงเทพมหานคร"]
#   index:       0          1          2       3        4          5
# gold entities: (1,2,'ORG')  and  (5,5,'LOC')

##HINT
# th_bio = ["O", "B-ORG", "I-ORG", "O", "O", "B-LOC"]
# th_bioes = bio_to_bioes(_______)
# recovered = bio_to_spans(______)

# TODO 1: write the BIO tags by hand (6 strings)
th_bio = None      # <-- e.g. ["O", "B-ORG", ...]

# TODO 2: convert to BIOES using bio_to_bioes()
th_bioes = None

# TODO 3: recover the spans and check they match the gold entities
recovered = None

print("BIO  :", th_bio)
print("BIOES:", th_bioes)
print("spans:", recovered)

BIO  : None
BIOES: None
spans: None


---
### 🔧 DIY 5 — Compute Precision, Recall, and F1

A system is run on a small test set. Here are the gold and predicted entity sets:

| | Entities |
|---|---|
| **Gold** | *สมชาย*/PER, *เชียงใหม่*/LOC, *ปตท.*/ORG, *กรุงเทพ*/LOC, *2026*/DATE |
| **Predicted** | *สมชาย*/PER, *เชียง*/LOC, *ปตท.*/ORG, *กรุงเทพ*/ORG |

Compute TP, FP, FN, then Precision, Recall, and F1 — **by hand first**, then check with code. Also: is precision or recall the bigger problem for this system, and what does that suggest you fix?

In [40]:
# === DIY 7 ===
gold_set = {("สมชาย", "PER"), ("เชียงใหม่", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "LOC"), ("2026", "DATE")}
pred_set = {("สมชาย", "PER"), ("เชียง", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "ORG")}

##HINT 
# tp_set = __ & __
# fp_set = __ - __
# fn_set = __ - __
# tp, fp, fn = len(tp_set), len(fp_set), len(fn_set)

# precision = __ / (__ + __)
# recall = __ / (__ + __)
# f1 = 2 * __ * __ / (__ + __)

# TODO 1: compute the three counts
tp = None
fp = None
fn = None

# TODO 2: compute the three metrics
precision = None
recall = None
f1 = None

print(f"TP={tp} FP={fp} FN={fn}")
print(f"P={precision} R={recall} F1={f1}")

TP=None FP=None FN=None
P=None R=None F1=None
